# Ocean (buoy) climate assessment — 1986–2005 D03-pinned buoy core

Longest-available climatology of the D03-pinned buoy observation set (8 for T, 7 for W; buoy 46005 off-domain and excluded) and of each model, per buoy and pooled. Buoy records are sparse, so the climatology is the mean over each buoy's surviving (shared) years — `n_years` is small (T 4–10, W 8–14) and is reported per buoy.

**Data source (no re-parse):** `pickles/taylor_annuals.pkl` (v10) buoy annual arrays (D03-pinned, relaxed buoy gates, like-for-like shared years).

In [9]:
import os, sys, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
BASE=os.getcwd()
with open(os.path.join(BASE,'pickles','taylor_annuals.pkl'),'rb') as f: TA=pickle.load(f)
with open(os.path.join(BASE,'pickles','buoy_wind_tables.pkl'),'rb') as f: _w=pickle.load(f)
TRUTH=_w['truth']   # {buoy:(lon,lat)}
print('taylor version:', TA['_version'])
POOL_T=['46041','46050','46131','46132','46146','46204','46206','46207']
POOL_W=['46041','46131','46132','46146','46204','46206','46207']
MODELS=['D03','D02','D01','CanESM2','CanRCM4']
MNAME={'D03':'D3','D02':'D15','D01':'D75','CanESM2':'CanESM2','CanRCM4':'CanRCM4'}
CANON={'t':POOL_T,'wind':POOL_W}
for v,p in CANON.items(): print(v,'core n=',len(p))

taylor version: 10
t core n= 8
wind core n= 7


In [10]:
def climo(obs, mods, sids):
    """Per-site period climatology (mean over surviving years) + pooled means.
    Like-for-like: each model is masked to the obs surviving years for that site."""
    rows=[]
    for s in sids:
        if s not in obs: continue
        o = np.asarray(obs[s], float)
        ok = np.isfinite(o)
        rec = {'sid': s, 'n_years': int(ok.sum()),
               'obs': o[ok].mean() if ok.any() else np.nan}
        for m in MODELS:
            if s in mods[m]:
                mraw = np.asarray(mods[m][s], float)
                shared = ok & np.isfinite(mraw)          # obs surviving AND model valid
                rec['obs_'+m] = o[shared].mean() if shared.any() else np.nan
                mv = mraw[ok]                            # model over obs surviving years
                rec[m] = mv[np.isfinite(mv)].mean() if np.isfinite(mv).any() else np.nan
            else:
                rec['obs_'+m] = np.nan; rec[m] = np.nan
        rows.append(rec)
    df = pd.DataFrame(rows).set_index('sid')
    pooled = {'obs': df['obs'].mean()}
    for m in MODELS:
        mm = df['obs_'+m].notna() & df[m].notna()
        pooled[m] = df.loc[mm, m].mean() if mm.any() else np.nan
    return df, pooled

In [11]:
SITES={}; POOLED={}
for v in ['t','wind']:
    SITES[v],POOLED[v]=climo(TA[v][0],TA[v][1],CANON[v])
    print(f'{v:5s}: {len(SITES[v])} buoys, obs pooled = {POOLED[v]["obs"]:.3f}')

t    : 8 buoys, obs pooled = 10.693
wind : 7 buoys, obs pooled = 6.418


In [12]:
tab=[]
for v in ['t','wind']:
    d=SITES[v]; p=POOLED[v]
    for m in MODELS:
        mm=d['obs_'+m].notna()&d[m].notna()
        ob=d.loc[mm,'obs_'+m]; mod=d.loc[mm,m]
        r2=np.corrcoef(ob,mod)[0,1]**2 if len(ob)>2 and np.std(ob)>0 and np.std(mod)>0 else np.nan
        tab.append(dict(var=v,model=MNAME[m],obs=ob.mean(),model_mean=mod.mean(),
                        bias=mod.mean()-ob.mean(),rmse=np.sqrt(((mod-ob)**2).mean()),
                        r2=r2,n=int(mm.sum())))
PTAB=pd.DataFrame(tab)
print(PTAB.to_string(index=False))

 var   model       obs  model_mean      bias     rmse       r2  n
   t      D3 10.693177    9.787197 -0.905980 1.022236 0.408816  8
   t     D15 10.693177    9.769839 -0.923339 1.002981 0.542718  8
   t     D75 10.693177    8.981741 -1.711436 1.996509 0.021966  8
   t CanESM2 10.693177    9.230483 -1.462694 1.858685 0.082920  8
   t CanRCM4 10.693177    9.994257 -0.698920 0.821614 0.642064  8
wind      D3  6.417890    7.039127  0.621237 0.646585 0.975013  7
wind     D15  6.417890    6.836269  0.418379 0.652972 0.868489  7
wind     D75  6.417890    5.939549 -0.478342 1.404189 0.844187  7
wind CanESM2  6.417890    6.328079 -0.089811 0.688546 0.867401  7
wind CanRCM4  6.417890    7.531833  1.113943 1.264335 0.794035  7


In [13]:
# ---- per-buoy climatology + n_years (T) ----
SITES['t'][['n_years','obs','D03','CanESM2','CanRCM4']].to_string(float_format=lambda x:'%.3f'%x)

'       n_years    obs    D03  CanESM2  CanRCM4\nsid                                           \n46041        6 10.827  9.617   10.337   10.593\n46050        6 11.581 10.185   10.907   11.531\n46131       10 10.585 10.620    7.309    9.882\n46132        5 10.429  9.512    9.896   10.035\n46146        4 11.101 10.726    7.625    9.722\n46204        7  9.901  8.977    8.665    9.237\n46206        7 10.808  9.374    9.427    9.683\n46207        8 10.313  9.287    9.678    9.271'

In [14]:
OUT=os.path.join(BASE,'pickles','climate_buoy.pkl')
with open(OUT,'wb') as f:
    pickle.dump({'sites':SITES,'pooled':POOLED,'pooled_table':PTAB,
                 'n_buoys':{v:len(SITES[v]) for v in SITES}}, f)
print('wrote',OUT)

wrote /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/pickles/climate_buoy.pkl


In [15]:
# ---- figure: buoy obs climatology + D03 bias, cartopy PlateCarree (offline: gridlines, no coastlines) ----
import cartopy.crs as ccrs
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), subplot_kw={'projection': ccrs.PlateCarree()})
for j, (v, lbl) in enumerate({'t':'T (deg C)','wind':'W (m/s)'}.items()):
    d = SITES[v].copy()
    d['bias'] = d['D03'] - d['obs_D03']
    ok = d['obs'].notna(); dd = d[ok]
    LON = [TRUTH[i][0] for i in dd.index]; LAT = [TRUTH[i][1] for i in dd.index]
    ax = axes[j]
    sc = ax.scatter(LON, LAT, c=dd['obs'], cmap='RdBu_r', s=70, edgecolors='k',
                    linewidths=0.4, transform=ccrs.PlateCarree())
    for i, ix in enumerate(dd.index):
        ax.annotate(ix, (LON[i], LAT[i]), fontsize=7, xytext=(3,3), textcoords='offset points')
    ax.set_title(lbl + ' obs climatology (D03)')
    ax.gridlines(draw_labels=True, linewidth=0.3, color='grey', alpha=0.5, linestyle=':')
    ax.set_extent([-132, -115, 44, 55], crs=ccrs.PlateCarree())
    ax.set_aspect('equal')
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
OUTF = os.path.join(BASE, 'figures', 'climate', 'climate_buoy_map.png')
fig.savefig(OUTF, dpi=150, bbox_inches='tight'); print('saved', OUTF)
plt.close(fig)


saved /gpfs/fs7/dfo/hpcmc/comda/fs2_comda/amh001/AI_WORKDIR/NOTEBOOKS/figures/climate/climate_buoy_map.png


## Interpretation

- Buoy climatologies are short (n_years small) — treat as indicative, not 20-yr means. 
- Pooled obs vs model table is the ocean network-level climate comparison.

In [16]:
# ---- (b) emit a .tex snippet of the pooled obs-vs-model climatology table ----
import numpy as np
def cell_bias(x):
    if not np.isfinite(x): return 'NA'
    if x == 0: return '$0$'
    if x < 0:
        a = abs(x)
        return '$-%.2f$' % a
    return '$+%s$' % ('%.2f' % x)
def cell(x, nd=2):
    return 'NA' if not np.isfinite(x) else f'{x:.{nd}f}'
lines = []
lines.append('% Buoy climatology: pooled obs vs model (period means; bias = model - obs)')
lines.append(r'\begin{tabular}{l l r r r r}')
lines.append(r'\toprule')
lines.append('Var & Model & obs & bias & RMSE & R$^2$ \\ ')
lines.append(r'\midrule')
for _, r in PTAB.iterrows():
    lines.append(f"{r['var']:>4s} & {r['model']:>8s} & {cell(r['obs'])} & {cell_bias(r['bias'])} & {cell(r['rmse'])} & {cell(r['r2'])} \\ ")
lines.append(r'\bottomrule')
lines.append(r'\end{tabular}')
tex = '\n'.join(lines)
print(tex)
with open(os.path.join(BASE, 'report', 'climate_buoy_table.tex'), 'w') as f: f.write(tex + '\n')
print('wrote report/climate_buoy_table.tex')


% Buoy climatology: pooled obs vs model (period means; bias = model - obs)
\begin{tabular}{l l r r r r}
\toprule
Var & Model & obs & bias & RMSE & R$^2$ \ 
\midrule
   t &       D3 & 10.69 & $-0.91$ & 1.02 & 0.41 \ 
   t &      D15 & 10.69 & $-0.92$ & 1.00 & 0.54 \ 
   t &      D75 & 10.69 & $-1.71$ & 2.00 & 0.02 \ 
   t &  CanESM2 & 10.69 & $-1.46$ & 1.86 & 0.08 \ 
   t &  CanRCM4 & 10.69 & $-0.70$ & 0.82 & 0.64 \ 
wind &       D3 & 6.42 & $+0.62$ & 0.65 & 0.98 \ 
wind &      D15 & 6.42 & $+0.42$ & 0.65 & 0.87 \ 
wind &      D75 & 6.42 & $-0.48$ & 1.40 & 0.84 \ 
wind &  CanESM2 & 6.42 & $-0.09$ & 0.69 & 0.87 \ 
wind &  CanRCM4 & 6.42 & $+1.11$ & 1.26 & 0.79 \ 
\bottomrule
\end{tabular}
wrote report/climate_buoy_table.tex
